Imports:

In [76]:
import torch
import transformers

transformers.logging.set_verbosity_error()

## Part A

Getting the model (`DistilGPT2`)...

In [77]:
model_name = "distilbert/distilgpt2"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name, device_map='auto')
model = transformers.AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Then the paragraph I'll use is below (from _The Martian_)

Tokenizing it yields:

In [78]:
paragraph = r"""Teddy swiveled his chair and looked out the window to the sky beyond. Night was edging in. "What must it be like?” He pondered. "He's stuck out there. He thinks he's totally alone and that we all gave up on him. What kind of effect does that have on a man's psychology?" He turned back to Venkat. "I wonder what he's thinking right now." LOG ENTRY: SOL 61 How come Aquaman can control whales? They're mammals! Makes no sense"""
tokenizer(paragraph)

{'input_ids': [51, 21874, 1509, 425, 992, 465, 5118, 290, 3114, 503, 262, 4324, 284, 262, 6766, 3675, 13, 5265, 373, 1225, 2667, 287, 13, 366, 2061, 1276, 340, 307, 588, 30, 447, 251, 679, 16723, 1068, 13, 366, 1544, 338, 7819, 503, 612, 13, 679, 6834, 339, 338, 6635, 3436, 290, 326, 356, 477, 2921, 510, 319, 683, 13, 1867, 1611, 286, 1245, 857, 326, 423, 319, 257, 582, 338, 15119, 1701, 679, 2900, 736, 284, 9932, 41826, 13, 366, 40, 4240, 644, 339, 338, 3612, 826, 783, 526, 41605, 12964, 40405, 25, 36817, 8454, 1374, 1282, 11446, 10546, 460, 1630, 24635, 30, 1119, 821, 23426, 0, 27433, 645, 2565], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Then the perplexity of the sequence is 

In [79]:
tokens = tokenizer(paragraph, return_tensors="pt").to(model.device)
h = model(**tokens, labels=tokens.input_ids).loss

# huggingface loss uses ln, so use exp()
ppl = torch.e ** h.item()
ppl

49.106479599645056

Then I'll shuffle the sequence, which results in tokens:

In [80]:
torch.manual_seed(42)
shuffled_tokens = tokenizer(paragraph, return_tensors="pt").to(model.device)
n = shuffled_tokens['input_ids'].shape[-1]
shuffled_tokens['input_ids'] = shuffled_tokens['input_ids'][..., torch.randperm(n)]
tokenizer.decode(shuffled_tokens['input_ids'])

[' haveWhat on stuck right\'s sw psychology on.. chair whales looked that and we allging come Makes back gave Night that now SOL can him LOG the He They to. out there and\'re ENered." alone he He mammals of. 61 a totally He does turned to he effectiveeddy man must kind?HeT was�\'s\'sTRY " sky How be. thinking? beyond out What. Aqu pond thinks: " Ven no control� " wonder ed window it what up the!\'sled his senseI?"aman inkat like']

And model perplexity of:

In [81]:
ppl = 2 ** model(**shuffled_tokens, labels=shuffled_tokens["input_ids"]).loss.item()
ppl

521.4959165909193

This increase makes sense since the second one is essentially asking the model to predict outputs for a random sequence, whereas the original is asking the model to predict from text.

## Part B

Below is the output of greedy decoding:

In [82]:
prompt = "Once upon a time"
tokens = tokenizer(prompt, return_tensors="pt").to(model.device)

torch.manual_seed(42)
output = model.generate(**tokens, max_length=150, temperature=0.0)
print(tokenizer.decode(output[0]))

Once upon a time of war, the United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence


Then with sampling and the temperature options ($T = 0$ is excluded since that is an invalid value for temperature, and is the same as greedy decoding):

In [83]:
prompt = "Once upon a time"
tokens = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    for i in range(1, 6):
        temp = i*0.3
        output = model.generate(
            **tokens, max_length=150, temperature=temp, do_sample=True
        )
        print(f"Temperature: {temp}\n{tokenizer.decode(output[0])}\n")

Temperature: 0.3
Once upon a time, the only way to get to the end of the world is to get to the end of the world.”


The world is a place where people are able to live, work, and live.
The world is a place where people can live, work, and live.
The world is a place where people can live, work, and live.
The world is a place where people can live, work, and live.
The world is a place where people can live, work, and live.
The world is a place where people can live, work, and live.
The world is a place where people can live, work, and live.
The world is a place

Temperature: 0.6
Once upon a time, the idea of a new world was becoming increasingly apparent to me.

It was a time when I could find a way to live in a new world. In my mind, the world was a new world for me. It was a time when I could find a way to live in a new world.
I was born in the same city as my parents. I was born in the same city as my parents. I was born in the same city as my parents. I was born in the same city as m

The quality and diversity of the outputs seems to be the greatest with the middle temperatures. The lower temperatures and greedy decoding have coherent text, but very repetitive; on the other hand, the higher temperatures tend to avoid repetition, but are generally less coherent. 

Increasing the temperature also increased the diversity of the response: lower temperatures seem to gravitate towards saying something about the world, whereas the higher temperatures explored other topics (both $T=1.2$ and $T=1.5$ completed the prompt with narratives).